# Install Library & Upload Dataset

In [1]:
!pip install -q dash plotly pandas

import os
from google.colab import files

if not os.path.exists("instax_sales_transaction_data.csv"):
    print("Silakan upload file instax_sales_transaction_data.csv:")
    files.upload()
else:
    print("File instax_sales_transaction_data.csv sudah siap!")

File instax_sales_transaction_data.csv sudah siap!


# Kode Lengkap Dashboard Interaktif Fujifilm Instax

In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from dash import Dash, html, dcc, Input, Output, State, ctx

# ==========================================
# 1. LOAD & PREPARE DATA
# ==========================================
df = pd.read_csv("instax_sales_transaction_data.csv")
df["Tanggal"] = pd.to_datetime(df["Tanggal"])
df["YearMonth"] = df["Tanggal"].dt.to_period("M").astype(str)

# Map Indonesian categorical labels to English for clean display
CATEGORY_MAP = {
    "Kamera": "Camera",
    "Film": "Film",
    "Aksesoris": "Accessories"
}
PAYMENT_MAP = {
    "Transfer Bank": "Bank Transfer",
    "Kartu Kredit": "Credit Card",
    "Paylater": "PayLater",
    "E-Wallet (Gopay/OVO)": "E-Wallet (GoPay/OVO)"
}

df["Category"] = df["Kategori"].map(CATEGORY_MAP).fillna(df["Kategori"])
df["Payment_Method"] = df["Metode_Bayar"].map(PAYMENT_MAP).fillna(df["Metode_Bayar"])
df["Channel_Type"] = np.where(
    df["Lokasi_Toko"].isin(["Jakarta Official Store", "Surabaya Branch"]),
    "Offline",
    "Online"
)
df["Promo_Status"] = np.where(df["Diskon_IDR"] > 0, "Promo", "Regular")

MONTH_MAP = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
}
DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Fujifilm Instax Brand Color Palette
RED = "#D0021B"
RED_LIGHT = "#FDE8EA"
RED_SOFT = "rgba(208, 2, 27, 0.16)"
TEAL = "#00A896"
DARK = "#1B222A"
BG_MAIN = "#F2F4F7"
TEXT_DARK = "#111827"
TEXT_MUTED = "#6B7280"

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def fmt_idr(val):
    if abs(val) >= 1e9: return f"IDR {val/1e9:.2f}B"
    if abs(val) >= 1e6: return f"IDR {val/1e6:.1f}M"
    if abs(val) >= 1e3: return f"IDR {val/1e3:.1f}K"
    return f"IDR {val:,.0f}"

def fmt_num(val):
    if abs(val) >= 1e6: return f"{val/1e6:.2f}M"
    if abs(val) >= 1e3: return f"{val/1e3:.1f}K"
    return f"{val:,.0f}"

def filter_data(year, month, store):
    d = df if store == "ALL" else df[df["Lokasi_Toko"] == store]
    curr = d.copy()
    if year != "ALL": curr = curr[curr["Tahun"] == int(year)]
    if month != "ALL": curr = curr[curr["Bulan"] == int(month)]

    if year != "ALL" and month != "ALL":
        y, m = int(year), int(month)
        py, pm = (y - 1, 12) if m == 1 else (y, m - 1)
        prev = d[(d["Tahun"] == py) & (d["Bulan"] == pm)]
        ref_curr, comp_label = curr, f"vs {MONTH_MAP[pm]} {py}"
    elif year != "ALL":
        prev = d[d["Tahun"] == int(year) - 1]
        ref_curr, comp_label = curr, f"vs {int(year) - 1}"
    elif month != "ALL":
        pm = 12 if int(month) == 1 else int(month) - 1
        prev = d[d["Bulan"] == pm]
        ref_curr, comp_label = curr, f"vs {MONTH_MAP[pm]}"
    else:
        mid = d["Tanggal"].min() + (d["Tanggal"].max() - d["Tanggal"].min()) / 2
        prev, ref_curr = d[d["Tanggal"] <= mid], d[d["Tanggal"] > mid]
        comp_label = "vs 1st half period"
    return curr, prev, ref_curr, comp_label

def calc_growth(curr_val, prev_val):
    if not prev_val:
        return None
    return (curr_val - prev_val) / prev_val * 100

def make_sparkline(vals):
    vals = vals if len(vals) >= 2 else [0, 0]
    fig = go.Figure(go.Scatter(
        y=vals, mode="lines", line=dict(color=RED, width=2),
        fill="tozeroy", fillcolor="rgba(208, 2, 27, 0.14)", hoverinfo="skip"
    ))
    fig.update_layout(
        margin=dict(l=0, r=0, t=2, b=2), height=38, showlegend=False,
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        xaxis=dict(visible=False), yaxis=dict(visible=False)
    )
    return fig

def kpi_card(icon, title, val_str, g_pct, comp_lbl, spark_vals):
    if g_pct is None:
        g_el = html.Span("— no prior data", style={"color": TEXT_MUTED, "fontSize": "11px"})
    else:
        arrow = "▲" if g_pct >= 0 else "▼"
        color = "#B91C1C" if g_pct >= 0 else "#6B7280"
        g_el = html.Span(f"{arrow} {abs(g_pct):.1f}% {comp_lbl}", style={"color": color, "fontSize": "11px", "fontWeight": "600"})
    return html.Div(
        style={
            "background": "#FFFFFF", "borderRadius": "14px", "padding": "14px 16px",
            "boxShadow": "0 2px 10px rgba(17,24,39,0.06)", "display": "flex",
            "alignItems": "center", "justifyContent": "space-between", "gap": "10px",
            "flex": "1", "minWidth": "200px"
        },
        children=[
            html.Div(
                style={"display": "flex", "alignItems": "center", "gap": "12px"},
                children=[
                    html.Div(icon, style={
                        "width": "42px", "height": "42px", "borderRadius": "50%",
                        "background": RED_LIGHT, "color": RED, "display": "flex",
                        "alignItems": "center", "justifyContent": "center", "fontSize": "19px", "flexShrink": "0"
                    }),
                    html.Div([
                        html.Div(title, style={"fontSize": "12px", "color": TEXT_MUTED, "fontWeight": "500"}),
                        html.Div(val_str, style={"fontSize": "21px", "fontWeight": "800", "color": TEXT_DARK, "lineHeight": "1.2"}),
                        g_el
                    ])
                ]
            ),
            dcc.Graph(figure=make_sparkline(spark_vals), config={"displayModeBar": False}, style={"width": "72px", "height": "38px"})
        ]
    )

def make_donut(labels, values, colors, center_top, center_sub, show_legend=True, height=185):
    fig = go.Figure(go.Pie(
        labels=labels, values=values, hole=0.62, sort=False,
        marker=dict(colors=colors, line=dict(color="#FFFFFF", width=2)),
        textinfo="none", hovertemplate="<b>%{label}</b><br>%{value:,.0f} (%{percent})<extra></extra>"
    ))
    fig.update_layout(
        margin=dict(l=4, r=4, t=4, b=4), paper_bgcolor="rgba(0,0,0,0)",
        height=height, showlegend=show_legend,
        legend=dict(orientation="v", yanchor="middle", y=0.5, x=1.02, font=dict(size=11, color=TEXT_DARK)),
        annotations=[dict(
            text=f"<b>{center_top}</b><br><span style='font-size:10px;color:{TEXT_MUTED}'>{center_sub}</span>",
            x=0.5, y=0.5, showarrow=False, font=dict(size=14, color=TEXT_DARK)
        )]
    )
    return fig

def make_sales_trend(curr, prev, year, month):
    fig = go.Figure()
    if month != "ALL":
        g_curr = curr.groupby(curr["Tanggal"].dt.day)["Total_Penjualan"].sum().reset_index()
        x_curr, y_curr, x_title = g_curr["Tanggal"], g_curr["Total_Penjualan"], "Day of Month"
        if len(prev) > 0:
            g_prev = prev.groupby(prev["Tanggal"].dt.day)["Total_Penjualan"].sum().reset_index()
            fig.add_trace(go.Scatter(
                x=g_prev["Tanggal"], y=g_prev["Total_Penjualan"], name="Previous Period", mode="lines",
                line=dict(color="rgba(180, 80, 90, 0.45)", width=2, dash="dot"),
                hovertemplate="Day %{x}<br>Previous: IDR %{y:,.0f}<extra></extra>"
            ))
    elif year != "ALL":
        g_curr = curr.groupby("Bulan")["Total_Penjualan"].sum().reindex(range(1, 13)).dropna().reset_index()
        x_curr, y_curr, x_title = g_curr["Bulan"].map(MONTH_MAP), g_curr["Total_Penjualan"], f"Month ({year})"
        if len(prev) > 0:
            g_prev = prev.groupby("Bulan")["Total_Penjualan"].sum().reindex(g_curr["Bulan"]).fillna(0).reset_index()
            fig.add_trace(go.Scatter(
                x=g_prev["Bulan"].map(MONTH_MAP), y=g_prev["Total_Penjualan"], name=f"Year {int(year)-1}", mode="lines",
                line=dict(color="rgba(180, 80, 90, 0.45)", width=2, dash="dot"),
                hovertemplate="%{x}<br>Previous: IDR %{y:,.0f}<extra></extra>"
            ))
    else:
        g_curr = curr.groupby("YearMonth")["Total_Penjualan"].sum().reset_index()
        x_curr, y_curr, x_title = g_curr["YearMonth"], g_curr["Total_Penjualan"], "Period (2022–2025)"

    fig.add_trace(go.Scatter(
        x=x_curr, y=y_curr, name="This Period", mode="lines+markers",
        line=dict(color=RED, width=3, shape="spline"), marker=dict(size=5, color=RED),
        fill="tozeroy", fillcolor=RED_SOFT,
        hovertemplate="<b>%{x}</b><br>Revenue: IDR %{y:,.0f}<extra></extra>"
    ))
    if len(y_curr) > 0:
        max_idx = y_curr.idxmax()
        fig.add_annotation(
            x=x_curr.loc[max_idx], y=y_curr.loc[max_idx], text=f"<b>Peak: {fmt_idr(y_curr.loc[max_idx])}</b>",
            showarrow=True, arrowhead=2, ax=0, ay=-26, bgcolor=RED, font=dict(color="#FFFFFF", size=11), borderpad=4
        )
    fig.update_layout(
        margin=dict(l=10, r=12, t=22, b=28), paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        height=245, legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, font=dict(size=11)),
        xaxis=dict(title=x_title, showgrid=False, tickfont=dict(size=10, color=TEXT_MUTED), title_font=dict(size=10, color=TEXT_MUTED)),
        yaxis=dict(showgrid=True, gridcolor="#E5E7EB", tickfont=dict(size=10, color=TEXT_MUTED))
    )
    return fig

def build_top5_list(curr):
    top5 = curr.groupby(["Nama_Produk", "Category"], as_index=False).agg(Qty=("Qty", "sum")).sort_values("Qty", ascending=False).head(5)
    max_qty = top5["Qty"].max() if len(top5) > 0 else 1
    cat_icon = {"Camera": "📷", "Film": "🎞️", "Accessories": "🎒"}
    rows = []
    for idx, r in enumerate(top5.itertuples(), start=1):
        pct = (r.Qty / max_qty) * 100
        rows.append(html.Div(style={"marginBottom": "11px"}, children=[
            html.Div(style={"display": "flex", "alignItems": "center", "justifyContent": "space-between", "marginBottom": "4px"}, children=[
                html.Div(style={"display": "flex", "alignItems": "center", "gap": "8px"}, children=[
                    html.Span(f"{idx}", style={"fontWeight": "700", "color": TEXT_MUTED, "fontSize": "12px", "width": "14px"}),
                    html.Span(cat_icon.get(r.Category, "📦"), style={"fontSize": "14px"}),
                    html.Span(r.Nama_Produk, style={"fontWeight": "600", "fontSize": "12px", "color": TEXT_DARK})
                ]),
                html.Span(f"{r.Qty:,} units", style={"fontWeight": "700", "fontSize": "12px", "color": TEXT_DARK})
            ]),
            html.Div(style={"background": "#F3F4F6", "borderRadius": "999px", "height": "7px", "overflow": "hidden", "marginLeft": "22px"}, children=[
                html.Div(style={"width": f"{pct:.1f}%", "height": "100%", "background": RED, "borderRadius": "999px"})
            ])
        ]))
    return rows

# ==========================================
# 3. DASH APP LAYOUT
# ==========================================
app = Dash(__name__, suppress_callback_exceptions=True)

app.index_string = """<!DOCTYPE html>
<html>
<head>
    {%metas%}<title>Fujifilm Instax Sales Dashboard</title>{%favicon%}{%css%}
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap" rel="stylesheet">
    <style>
        * { box-sizing: border-box; font-family: 'Inter', sans-serif; }
        body { margin: 0; background: #F2F4F7; }
        .nav-btn:hover { background: rgba(255,255,255,0.08) !important; }
        .Select-control { border-radius: 10px !important; border: 1px solid #D1D5DB !important; font-size: 12px !important; }
    </style>
</head>
<body>{%app_entry%}<footer>{%config%}{%scripts%}{%renderer%}</footer></body>
</html>"""

MENU_ITEMS = [
    ("overview", "🏠", "Overview"),
    ("sales", "📈", "Sales"),
    ("products", "📷", "Products"),
    ("channels", "🏪", "Channels"),
    ("payments", "💳", "Payments"),
]

def nav_style(active=False):
    return {
        "width": "100%", "textAlign": "left", "padding": "10px 14px", "marginBottom": "6px",
        "border": "none", "borderRadius": "10px", "cursor": "pointer", "fontSize": "13px",
        "fontWeight": "600" if active else "500", "background": RED if active else "transparent",
        "color": "#FFFFFF" if active else "#CBD5E1", "display": "flex", "alignItems": "center", "gap": "10px"
    }

year_opts = [{"label": "All Years (2022–2025)", "value": "ALL"}] + [{"label": str(y), "value": str(y)} for y in sorted(df["Tahun"].unique())]
month_opts = [{"label": "All Months", "value": "ALL"}] + [{"label": MONTH_MAP[m], "value": str(m)} for m in range(1, 13)]
store_opts = [{"label": "All Stores / Channels", "value": "ALL"}] + [{"label": s, "value": s} for s in sorted(df["Lokasi_Toko"].unique())]

app.layout = html.Div(
    style={"display": "flex", "minHeight": "790px", "background": BG_MAIN, "padding": "10px", "gap": "12px"},
    children=[
        dcc.Store(id="active-page", data="overview"),
        # Left Sidebar
        html.Div(
            style={
                "width": "195px", "background": DARK, "borderRadius": "18px", "padding": "22px 14px",
                "display": "flex", "flexDirection": "column", "justifyContent": "space-between",
                "flexShrink": "0", "boxShadow": "0 4px 16px rgba(0,0,0,0.18)"
            },
            children=[
                html.Div([
                    html.Div(style={"paddingLeft": "8px", "marginBottom": "28px"}, children=[
                        html.Span("FUJ", style={"color": "#FFFFFF", "fontWeight": "900", "fontSize": "22px"}),
                        html.Span("I", style={"color": RED, "fontWeight": "900", "fontSize": "22px"}),
                        html.Span("FILM", style={"color": "#FFFFFF", "fontWeight": "900", "fontSize": "22px"}),
                        html.Div("INSTAX ANALYTICS", style={"color": "#94A3B8", "fontSize": "9px", "letterSpacing": "2px"})
                    ]),
                    html.Div([
                        html.Button([html.Span(icon), html.Span(label)], id=f"btn-{key}", n_clicks=0, className="nav-btn", style=nav_style(key == "overview"))
                        for key, icon, label in MENU_ITEMS
                    ])
                ]),
                html.Div(style={"padding": "12px 8px", "borderTop": "1px solid rgba(255,255,255,0.1)"}, children=[
                    html.Div("Focus on", style={"color": "#FFFFFF", "fontWeight": "700", "fontSize": "14px"}),
                    html.Div("what matters", style={"color": RED, "fontWeight": "700", "fontSize": "14px"}),
                ])
            ]
        ),
        # Main Workspace
        html.Div(
            style={"flex": "1", "display": "flex", "flexDirection": "column", "gap": "12px", "minWidth": "0"},
            children=[
                html.Div(
                    style={"display": "flex", "flexWrap": "wrap", "alignItems": "center", "justifyContent": "space-between", "gap": "12px", "padding": "4px"},
                    children=[
                        html.Div([
                            html.H2("Fujifilm Instax Sales Dashboard", style={"margin": "0", "fontSize": "22px", "fontWeight": "800", "color": TEXT_DARK}),
                            html.Div("Track sales performance and uncover actionable insights for business growth", style={"fontSize": "12px", "color": TEXT_MUTED})
                        ]),
                        html.Div(style={"display": "flex", "gap": "8px", "flexWrap": "wrap"}, children=[
                            html.Div([html.Div("Year", style={"fontSize": "10px", "fontWeight": "700", "color": RED}), dcc.Dropdown(id="filter-year", options=year_opts, value="ALL", clearable=False, style={"width": "165px"})]),
                            html.Div([html.Div("Month", style={"fontSize": "10px", "fontWeight": "700", "color": TEXT_MUTED}), dcc.Dropdown(id="filter-month", options=month_opts, value="ALL", clearable=False, style={"width": "130px"})]),
                            html.Div([html.Div("Store / Channel", style={"fontSize": "10px", "fontWeight": "700", "color": TEXT_MUTED}), dcc.Dropdown(id="filter-store", options=store_opts, value="ALL", clearable=False, style={"width": "185px"})]),
                        ])
                    ]
                ),
                html.Div(id="main-content", style={"display": "flex", "flexDirection": "column", "gap": "12px"})
            ]
        )
    ]
)

# ==========================================
# 4. CALLBACKS
# ==========================================
@app.callback(
    Output("active-page", "data"),
    [Output(f"btn-{k}", "style") for k, _, _ in MENU_ITEMS],
    [Input(f"btn-{k}", "n_clicks") for k, _, _ in MENU_ITEMS],
    State("active-page", "data")
)
def switch_page(*args):
    current_page = args[-1]
    if ctx.triggered_id and ctx.triggered_id.startswith("btn-"):
        current_page = ctx.triggered_id.replace("btn-", "")
    return current_page, *[nav_style(k == current_page) for k, _, _ in MENU_ITEMS]

@app.callback(
    Output("main-content", "children"),
    Input("active-page", "data"),
    Input("filter-year", "value"),
    Input("filter-month", "value"),
    Input("filter-store", "value")
)
def update_dashboard(page, year, month, store):
    year, month, store = year or "ALL", month or "ALL", store or "ALL"
    curr, prev, ref_curr, comp_label = filter_data(year, month, store)

    if len(curr) == 0:
        return html.Div(
            style={"background": "#FFFFFF", "borderRadius": "14px", "padding": "48px", "textAlign": "center"},
            children=[
                html.H3("No transaction data available for this filter selection", style={"color": RED}),
                html.P("Available dataset spans from May 1, 2022 to May 1, 2025. Please select another month or year.", style={"color": TEXT_MUTED})
            ]
        )

    rev, orders, qty = curr["Total_Penjualan"].sum(), len(curr), curr["Qty"].sum()
    aov = rev / orders
    rev_g = calc_growth(ref_curr["Total_Penjualan"].sum(), prev["Total_Penjualan"].sum())
    ord_g = calc_growth(len(ref_curr), len(prev))
    qty_g = calc_growth(ref_curr["Qty"].sum(), prev["Qty"].sum())
    prev_aov = (prev["Total_Penjualan"].sum() / len(prev)) if len(prev) > 0 else 0
    ref_aov = (ref_curr["Total_Penjualan"].sum() / len(ref_curr)) if len(ref_curr) > 0 else 0
    aov_g = calc_growth(ref_aov, prev_aov)

    grp_col = curr["Tanggal"].dt.day if month != "ALL" else curr["YearMonth"]
    sp_rev = curr.groupby(grp_col)["Total_Penjualan"].sum().values
    sp_ord = curr.groupby(grp_col).size().values
    sp_qty = curr.groupby(grp_col)["Qty"].sum().values
    sp_aov = sp_rev / np.maximum(sp_ord, 1)

    kpi_row = html.Div(style={"display": "flex", "gap": "12px", "flexWrap": "wrap"}, children=[
        kpi_card("🛍️", "Total Revenue", fmt_idr(rev), rev_g, comp_label, sp_rev),
        kpi_card("🛒", "Total Orders", fmt_num(orders), ord_g, comp_label, sp_ord),
        kpi_card("📦", "Products Sold", fmt_num(qty), qty_g, comp_label, sp_qty),
        kpi_card("⭐", "Avg. Order Value (AOV)", fmt_idr(aov), aov_g, comp_label, sp_aov),
    ])

    card_box = {"background": "#FFFFFF", "borderRadius": "14px", "padding": "14px 16px", "boxShadow": "0 2px 10px rgba(17,24,39,0.06)"}

    if page == "overview":
        cat_rev = curr.groupby("Category")["Total_Penjualan"].sum().sort_values(ascending=False)
        prod_qty = curr.groupby("Nama_Produk")["Qty"].sum().sort_values(ascending=False)
        online_pct = (curr["Channel_Type"] == "Online").mean() * 100
        promo_pct = (curr["Promo_Status"] == "Promo").mean() * 100
        top_pay = curr["Payment_Method"].value_counts().index[0]
        top_pay_pct = curr["Payment_Method"].value_counts(normalize=True).iloc[0] * 100

        growth_txt = (
            f"Revenue {'increased' if rev_g >= 0 else 'decreased'} by {abs(rev_g):.1f}% ({fmt_idr(rev)}) {comp_label}, with an average order value of {fmt_idr(aov)}."
            if rev_g is not None else f"Total revenue reached {fmt_idr(rev)} across {orders:,} orders with an AOV of {fmt_idr(aov)}."
        )

        row2 = html.Div(style={"display": "grid", "gridTemplateColumns": "1.65fr 1.2fr 0.95fr", "gap": "12px"}, children=[
            html.Div(style=card_box, children=[
                html.Div("📊 Sales Trend", style={"fontWeight": "700", "fontSize": "14px", "marginBottom": "4px"}),
                dcc.Graph(figure=make_sales_trend(curr, prev, year, month), config={"displayModeBar": False})
            ]),
            html.Div(style=card_box, children=[
                html.Div("🏆 Top 5 Best-Selling Products", style={"fontWeight": "700", "fontSize": "14px", "marginBottom": "14px"}),
                html.Div(build_top5_list(curr))
            ]),
            html.Div(style={"background": DARK, "borderRadius": "14px", "overflow": "hidden", "display": "flex", "flexDirection": "column"}, children=[
                html.Div("💡 Key Insights", style={"background": RED, "color": "#FFFFFF", "fontWeight": "700", "fontSize": "14px", "padding": "11px 15px"}),
                html.Div(style={"padding": "14px 15px", "color": "#E5E7EB", "fontSize": "12px", "lineHeight": "1.55", "display": "flex", "flexDirection": "column", "gap": "10px"}, children=[
                    html.Div([html.Span("✓ ", style={"color": "#F87171", "fontWeight": "800"}), growth_txt]),
                    html.Div([html.Span("✓ ", style={"color": "#F87171", "fontWeight": "800"}), f"The {cat_rev.index[0]} category contributed {cat_rev.iloc[0]/rev*100:.1f}% of revenue; top-selling item is {prod_qty.index[0]} ({prod_qty.iloc[0]:,} units)."]),
                    html.Div([html.Span("✓ ", style={"color": "#F87171", "fontWeight": "800"}), f"Online channels drove {online_pct:.1f}% of orders, led by {top_pay} ({top_pay_pct:.1f}%), with {promo_pct:.1f}% promo transactions."]),
                ])
            ])
        ])

        cat_df = cat_rev.reset_index()
        ch_counts = curr["Channel_Type"].value_counts().reindex(["Online", "Offline"], fill_value=0)
        pr_counts = curr["Promo_Status"].value_counts().reindex(["Promo", "Regular"], fill_value=0)
        pay_df = curr["Payment_Method"].value_counts().reset_index(); pay_df.columns = ["Label", "Count"]

        row3 = html.Div(style={"display": "grid", "gridTemplateColumns": "1fr 1.25fr 1fr", "gap": "12px"}, children=[
            html.Div(style=card_box, children=[
                html.Div("Sales by Category", style={"fontWeight": "700", "fontSize": "14px", "marginBottom": "4px"}),
                dcc.Graph(figure=make_donut(cat_df["Category"], cat_df["Total_Penjualan"], [RED, TEAL, "#374151"], f"{cat_df.iloc[0]['Total_Penjualan']/rev*100:.1f}%", cat_df.iloc[0]["Category"]), config={"displayModeBar": False})
            ]),
            html.Div(style=card_box, children=[
                html.Div("👥 Channel & Promo Analysis", style={"fontWeight": "700", "fontSize": "14px", "marginBottom": "4px"}),
                html.Div(style={"display": "grid", "gridTemplateColumns": "1fr 1fr", "gap": "4px"}, children=[
                    html.Div([
                        html.Div("Online vs Offline", style={"fontSize": "11px", "color": TEXT_MUTED, "textAlign": "center", "fontWeight": "600"}),
                        dcc.Graph(figure=make_donut(ch_counts.index, ch_counts.values, [RED, "#374151"], f"{online_pct:.1f}%", "Online", False, 165), config={"displayModeBar": False})
                    ]),
                    html.Div([
                        html.Div("Promo vs Regular", style={"fontSize": "11px", "color": TEXT_MUTED, "textAlign": "center", "fontWeight": "600"}),
                        dcc.Graph(figure=make_donut(pr_counts.index, pr_counts.values, [RED, "#374151"], f"{promo_pct:.1f}%", "Promo", False, 165), config={"displayModeBar": False})
                    ])
                ])
            ]),
            html.Div(style=card_box, children=[
                html.Div("💳 Payment Methods", style={"fontWeight": "700", "fontSize": "14px", "marginBottom": "4px"}),
                dcc.Graph(figure=make_donut(pay_df["Label"], pay_df["Count"], [RED, "#374151", TEAL, "#9CA3AF"], "100%", f"{orders:,} Orders"), config={"displayModeBar": False})
            ])
        ])
        return [kpi_row, row2, row3]

    if page == "sales":
        by_day = curr.groupby("Hari")["Total_Penjualan"].sum().reindex(DAY_ORDER).fillna(0).reset_index()
        fig_day = px.bar(by_day, x="Hari", y="Total_Penjualan", color_discrete_sequence=[RED], labels={"Hari": "Day of Week", "Total_Penjualan": "Revenue (IDR)"})
        fig_day.update_layout(margin=dict(l=10, r=10, t=20, b=20), paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", height=280)
        by_yr = curr.groupby(["Tahun", "Category"], as_index=False)["Total_Penjualan"].sum()
        fig_yr = px.bar(by_yr, x="Tahun", y="Total_Penjualan", color="Category", barmode="group", color_discrete_map={"Camera": RED, "Film": TEAL, "Accessories": "#374151"}, labels={"Tahun": "Year", "Total_Penjualan": "Revenue (IDR)"})
        fig_yr.update_layout(margin=dict(l=10, r=10, t=20, b=20), paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", height=280)
        return [kpi_row, html.Div(style={"display": "grid", "gridTemplateColumns": "1fr 1fr", "gap": "12px"}, children=[
            html.Div(style=card_box, children=[html.Div("📅 Revenue by Day of Week", style={"fontWeight": "700", "fontSize": "14px"}), dcc.Graph(figure=fig_day, config={"displayModeBar": False})]),
            html.Div(style=card_box, children=[html.Div("📆 Annual Revenue by Category", style={"fontWeight": "700", "fontSize": "14px"}), dcc.Graph(figure=fig_yr, config={"displayModeBar": False})])
        ])]

    if page == "products":
        prod_df = curr.groupby(["Nama_Produk", "Category"], as_index=False).agg(Revenue=("Total_Penjualan", "sum")).sort_values("Revenue")
        fig_prod = px.bar(prod_df, x="Revenue", y="Nama_Produk", color="Category", orientation="h", color_discrete_map={"Camera": RED, "Film": TEAL, "Accessories": "#374151"}, labels={"Nama_Produk": "Product", "Revenue": "Revenue (IDR)"})
        fig_prod.update_layout(margin=dict(l=10, r=10, t=15, b=20), paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", height=380)
        return [kpi_row, html.Div(style=card_box, children=[html.Div("📷 Full Product Revenue Ranking", style={"fontWeight": "700", "fontSize": "14px"}), dcc.Graph(figure=fig_prod, config={"displayModeBar": False})])]

    if page == "channels":
        st_df = curr.groupby(["Lokasi_Toko", "Channel_Type"], as_index=False).agg(Revenue=("Total_Penjualan", "sum")).sort_values("Revenue", ascending=False)
        fig_st = px.bar(st_df, x="Lokasi_Toko", y="Revenue", color="Channel_Type", color_discrete_map={"Online": RED, "Offline": "#374151"}, labels={"Lokasi_Toko": "Store / Platform", "Revenue": "Revenue (IDR)", "Channel_Type": "Channel"})
        fig_st.update_layout(margin=dict(l=10, r=10, t=15, b=20), paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", height=360)
        return [kpi_row, html.Div(style=card_box, children=[html.Div("🏪 Revenue by Store / Platform", style={"fontWeight": "700", "fontSize": "14px"}), dcc.Graph(figure=fig_st, config={"displayModeBar": False})])]

    pay_sum = curr.groupby("Payment_Method", as_index=False).agg(Revenue=("Total_Penjualan", "sum")).sort_values("Revenue", ascending=False)
    fig_pay = px.bar(pay_sum, x="Payment_Method", y="Revenue", text=pay_sum["Revenue"].apply(fmt_idr), color="Payment_Method", color_discrete_sequence=[RED, "#374151", TEAL, "#9CA3AF"], labels={"Payment_Method": "Payment Method", "Revenue": "Revenue (IDR)"})
    fig_pay.update_layout(showlegend=False, margin=dict(l=10, r=10, t=20, b=20), paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", height=360)
    return [kpi_row, html.Div(style=card_box, children=[html.Div("💳 Total Revenue by Payment Method", style={"fontWeight": "700", "fontSize": "14px"}), dcc.Graph(figure=fig_pay, config={"displayModeBar": False})])]

# ==========================================
# 5. RUN IN GOOGLE COLAB
# ==========================================
app.run(jupyter_mode="inline", port=8050, jupyter_height=830)
# === END ===

<IPython.core.display.Javascript object>